# 微信聊天角色扮演 - Qwen3-4B 微调

In [ ]:
# 安装依赖
! pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 检查GPU
print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# data

In [ ]:
from google.colab import files

# 上传训练数据（包含 system_prompt 和对话数据的合并文件）
uploaded = files.upload()
data_file = list(uploaded.keys())[0]
print(f"已上传: {data_file}")

In [ ]:
# 加载合并文件（system_prompt + 训练数据）
with open(data_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

system_prompt = data.get("system_prompt", "")
train_data = data["conversations"]

# 注入 system prompt 到每条对话开头
if system_prompt:
    for sample in train_data:
        conversations = sample["conversations"]
        if conversations[0]["role"] != "system":
            conversations.insert(0, {"role": "system", "content": system_prompt})
    print(f"已注入 system prompt: {system_prompt}")
else:
    print("未设置 system prompt, 跳过注入")

In [ ]:
# 将 conversations 重命名为 messages（TRL 要求的列名）
for sample in train_data:
    sample["messages"] = sample.pop("conversations")

dataset = Dataset.from_list(train_data)

print(f"数据集大小: {len(dataset)}")
print(f"\n样本示例:")
print(json.dumps(dataset[0]["messages"], ensure_ascii=False, indent=2))

# model and tokenizer

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# 设置pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"词表大小: {len(tokenizer)}")
print(f"特殊token: pad={tokenizer.pad_token}, eos={tokenizer.eos_token}")

In [ ]:
# 为 chat template 添加 {% generation %} 标记，支持 TRL assistant_only_loss
# 修改点：assistant 分支中，在 content 前加 {% generation %}，在 <|im_end|> 后加 {% endgeneration %}

tokenizer.chat_template = (
    "{%- if tools %}\n"
    "    {{- '<|im_start|>system\\n' }}\n"
    "    {%- if messages[0].role == 'system' %}\n"
    "        {{- messages[0].content + '\\n\\n' }}\n"
    "    {%- endif %}\n"
    '    {{- "# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n'
    "    {%- for tool in tools %}\n"
    '        {{- "\\n" }}\n'
    "        {{- tool | tojson }}\n"
    "    {%- endfor %}\n"
    '    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n'
    "{%- else %}\n"
    "    {%- if messages[0].role == 'system' %}\n"
    "        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}\n"
    "    {%- endif %}\n"
    "{%- endif %}\n"
    "{%- for message in messages %}\n"
    "    {%- if message.content is string %}\n"
    "        {%- set content = message.content %}\n"
    "    {%- else %}\n"
    "        {%- set content = '' %}\n"
    "    {%- endif %}\n"
    '    {%- if (message.role == "user") or (message.role == "system" and not loop.first) %}\n'
    "        {{- '<|im_start|>' + message.role + '\\n' + content + '<|im_end|>' + '\\n' }}\n"
    '    {%- elif message.role == "assistant" %}\n'
    "        {{- '<|im_start|>' + message.role + '\\n' }}{% generation %}{{- content }}\n"
    "        {%- if message.tool_calls %}\n"
    "            {%- for tool_call in message.tool_calls %}\n"
    "                {%- if (loop.first and content) or (not loop.first) %}\n"
    "                    {{- '\\n' }}\n"
    "                {%- endif %}\n"
    "                {%- if tool_call.function %}\n"
    "                    {%- set tool_call = tool_call.function %}\n"
    "                {%- endif %}\n"
    '                {{- \'<tool_call>\\n{"name": "\' }}\n'
    "                {{- tool_call.name }}\n"
    '                {{- \'", "arguments": \' }}\n'
    "                {%- if tool_call.arguments is string %}\n"
    "                    {{- tool_call.arguments }}\n"
    "                {%- else %}\n"
    "                    {{- tool_call.arguments | tojson }}\n"
    "                {%- endif %}\n"
    "                {{- '}\\n</tool_call>' }}\n"
    "            {%- endfor %}\n"
    "        {%- endif %}\n"
    "        {{- '<|im_end|>\\n' }}{% endgeneration %}\n"
    '    {%- elif message.role == "tool" %}\n'
    '        {%- if loop.first or (messages[loop.index0 - 1].role != "tool") %}\n'
    "            {{- '<|im_start|>user' }}\n"
    "        {%- endif %}\n"
    "        {{- '\\n<tool_response>\\n' }}\n"
    "        {{- content }}\n"
    "        {{- '\\n</tool_response>' }}\n"
    '        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}\n'
    "            {{- '<|im_end|>\\n' }}\n"
    "        {%- endif %}\n"
    "    {%- endif %}\n"
    "{%- endfor %}\n"
    "{%- if add_generation_prompt %}\n"
    "    {{- '<|im_start|>assistant\\n' }}\n"
    "{%- endif %}\n"
)

print("✓ chat template 已设置（含 {% generation %} 标记）")

# 验证：文本输出不变
sample = [
    {"role": "system", "content": "你是一个微信聊天对象"},
    {"role": "user", "content": "在干嘛"},
    {"role": "assistant", "content": "刚下班 累死了"},
]
print("\n=== 验证训练格式 ===")
print(tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False))
print("=== 验证推理格式 ===")
print(tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True))

In [ ]:
# 4-bit量化配置（节省显存）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # Qwen原生BF16
    bnb_4bit_use_double_quant=True
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 准备模型进行k-bit训练
model = prepare_model_for_kbit_training(model)

print(f"模型加载完成")
print(f"模型参数量: {model.num_parameters() / 1e6:.1f}M")

# LoRA

In [ ]:
# LoRA配置
# 数据量极小（29条对话），r=4 足够，避免高容量过拟合
lora_config = LoraConfig(
    r=4,                     # 从8降至4：29条数据不需要高秩
    lora_alpha=8,            # 保持 alpha/r = 2
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# train

In [ ]:
# 训练参数
# 针对极小数据集（~20条对话）防过拟合：适度epoch、低学习率
training_args = SFTConfig(
    output_dir="./output",
    num_train_epochs=5,                  # 20条数据需要多过几遍才能学到风格
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,       # 有效batch=4，每epoch更新5步，学习更充分
    learning_rate=5e-5,                  # 2e-4→5e-5：最关键，学习率过高是崩坏主因
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                    # 总步数少，warmup 占比应更大
    bf16=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=5,                        # 每5步保存，便于回退到过拟合前的checkpoint
    save_total_limit=5,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    packing=False,
    assistant_only_loss=True,
)

tokenizer.model_max_length = 512         # 避免截断长对话，数据量极小不影响显存

# 估算步数
n_samples = len(dataset)                 # 动态获取，不再硬编码
steps_per_epoch = max(1, n_samples // training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs
warmup_steps = max(1, int(total_steps * training_args.warmup_ratio))

print("训练配置:")
print(f"  样本数: {n_samples}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Steps per epoch (estimated): {steps_per_epoch}")
print(f"  Total steps (estimated): {total_steps}")
print(f"  Warmup steps (estimated): {warmup_steps}")
print()
print("过拟合风险提示：训练时观察 loss, 若 loss < 0.3 应提前停止并使用更早的 checkpoint")

In [ ]:
# 创建Trainer
# 直接传入 messages 列，TRL 自动应用 chat template
# assistant_only_loss=True 确保仅对 assistant 回复计算 loss
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer已创建, 准备开始训练")

In [ ]:
# ========== 诊断：排查问题出在哪个环节 ==========

# 1) 训练前测试：模型加载+LoRA后（未训练），是否还能正常对话？
#    如果这一步就输出乱码，说明问题在 quantization/模型加载/chat template，不是训练
print("=== 诊断1: 训练前模型测试 ===")
test_msgs = [{"role": "user", "content": "你好"}]
if system_prompt:
    test_msgs = [{"role": "system", "content": system_prompt}] + test_msgs
test_prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**test_inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.pad_token_id)
generated = out[0][test_inputs["input_ids"].shape[1]:]
print(f"训练前输出: {tokenizer.decode(generated, skip_special_tokens=True)}")
print()

# 2) 检查 TRL 处理后的训练数据：token 和 label 是否正确？
#    label=-100 表示不计算loss，非-100表示要学习的token
print("=== 诊断2: TRL 处理后的训练数据 ===")
processed = trainer.train_dataset[0]
input_ids = processed["input_ids"]
labels = processed["labels"]
print(f"序列长度: {len(input_ids)}")

# 统计 loss token 数量
loss_token_count = sum(1 for l in labels if l != -100)
total_token_count = len(labels)
print(f"计算loss的token: {loss_token_count}/{total_token_count} ({loss_token_count/total_token_count*100:.1f}%)")

# 显示哪些token在计算loss
print("\n--- 有loss的token (应该只是assistant回复) ---")
for i, (tid, lid) in enumerate(zip(input_ids, labels)):
    if lid != -100:
        print(f"  pos {i}: token={tokenizer.decode([tid])} (id={tid}, label={lid})")
    if i > 5 and lid == -100 and i < len(labels) - 1:
        # 只在边界处打印
        pass

# 简化版：显示完整序列，标记哪些部分有loss
print("\n--- 完整序列（★=有loss的部分）---")
chunks = []
current_chunk = ""
current_has_loss = (labels[0] != -100)
for tid, lid in zip(input_ids, labels):
    has_loss = (lid != -100)
    tok_text = tokenizer.decode([tid])
    if has_loss == current_has_loss:
        current_chunk += tok_text
    else:
        prefix = "★" if current_has_loss else " "
        chunks.append(f"{prefix} {repr(current_chunk)}")
        current_chunk = tok_text
        current_has_loss = has_loss
prefix = "★" if current_has_loss else " "
chunks.append(f"{prefix} {repr(current_chunk)}")
for chunk in chunks:
    print(chunk)

In [ ]:
# 开始训练
print("开始训练...")
trainer.train()
print("训练完成！")

# test

In [ ]:
import os

# 默认使用训练结束时的模型进行测试
# 如果发现胡言乱语，可以改成更早的 checkpoint，例如 "./output/checkpoint-5"
CHECKPOINT_TO_TEST = None   # None = 使用当前模型；填路径 = 加载指定 checkpoint

if CHECKPOINT_TO_TEST and os.path.exists(CHECKPOINT_TO_TEST):
    from peft import PeftModel
    print(f"从 checkpoint 加载 LoRA: {CHECKPOINT_TO_TEST}")
    # 重用已加载的基座（无需重新下载），只替换 LoRA 权重
    merged_model = PeftModel.from_pretrained(model.base_model.model, CHECKPOINT_TO_TEST)
    print(f"已加载 checkpoint: {CHECKPOINT_TO_TEST}")
else:
    merged_model = model
    print("使用训练结束时的模型进行测试")
    print("提示：如果结果不好，修改上方 CHECKPOINT_TO_TEST 换到更早的 checkpoint 重新运行此格")

In [ ]:
def chat(model, tokenizer, user_input, history=[]):
    """
    单轮对话测试
    """
    messages = history + [{"role": "user", "content": user_input}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()

In [ ]:
# 测试对话
test_inputs = [
    "在干嘛",
    "晚上吃什么",
    "一起看电影吧"
]

# 使用与训练一致的 system prompt
test_history = []
if system_prompt:
    test_history = [{"role": "system", "content": system_prompt}]

print("=== 模型测试 ===")
if system_prompt:
    print(f"[system prompt: {system_prompt}]\n")
for user_input in test_inputs:
    response = chat(merged_model, tokenizer, user_input, history=test_history)
    print(f"用户: {user_input}")
    print(f"模型: {response}")
    print()

# download

将训练好的模型下载到本地

In [ ]:
# 保存LoRA适配器
LORA_OUTPUT_DIR = "./lora_adapter"
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"LoRA适配器已保存至: {LORA_OUTPUT_DIR}")

In [ ]:
# 打包LoRA适配器
! zip -r lora_adapter.zip ./lora_adapter

# 下载
from google.colab import files
files.download('lora_adapter.zip')

print("LoRA适配器已下载，解压后放到本地项目目录使用")

In [ ]:
# （可选）打包合并后的完整模型
# 注意：完整模型文件较大，下载可能需要较长时间
! zip -r merged_model.zip ./merged_model

from google.colab import files
files.download('merged_model.zip')